# ==========================================================
# 02_load_parallel_validate.ipynb - Loading & Normalization
# ==========================================================

In [ ]:
import gzip
import json
import pathlib
import pandas as pd

In [ ]:
# Define project folder paths
PROJECT_ROOT = pathlib.Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ----------------------------------------------------------
# Step 1: Load and Clean Patients Master Data
# ----------------------------------------------------------
patients_file_path = RAW_DIR / "patients" / "patients.csv.gz"
df_patients = pd.read_csv(patients_file_path, dtype=str)

print("Patients Dataset Initial Shape:", df_patients.shape)
print(df_patients.head())

In [ ]:
# Normalize phone numbers (extract last 10 digits)
df_patients["phone_norm"] = df_patients["phone"].fillna("").str.replace(r"\D", "", regex=True).str[-10:]

# Normalize patient names (uppercase and single spaces)
df_patients["name_norm"] = df_patients["patient_name"].fillna("").str.upper().str.replace(r"\s+", " ", regex=True).str.strip()

# Generate unique person key to identify duplicate records
df_patients["person_key"] = df_patients.groupby(["phone_norm", "name_norm", "dob"]).ngroup()

# Remove duplicate rows keeping first instance
df_patients_clean = df_patients.drop_duplicates(subset=["person_key"], keep="first").copy()

print(f"Deduplicated Patients: {len(df_patients):,} rows -> {len(df_patients_clean):,} unique people")

In [ ]:
# Save cleaned patients data
df_patients_clean.to_csv(CLEAN_DIR / "patients_clean.csv.gz", index=False, compression="gzip")
print("Saved cleaned patients to data/clean/patients_clean.csv.gz")

In [ ]:
# ----------------------------------------------------------
# Step 2: Merge Legacy and Nova Bookings Data
# ----------------------------------------------------------
# Fetch all legacy files
legacy_files = sorted((RAW_DIR / "bookings" / "legacy").rglob("*.csv.gz"))
print(f"Found {len(legacy_files)} legacy files")

list_of_legacy_dfs = []
for file_path in legacy_files:
    temp_df = pd.read_csv(file_path, dtype=str)
    temp_df["source_file"] = file_path.name
    temp_df["system"] = "legacy"
    list_of_legacy_dfs.append(temp_df)

df_all_legacy = pd.concat(list_of_legacy_dfs, ignore_index=True)

# Normalize city names
df_all_legacy["city"] = df_all_legacy["city"].str.strip().replace({"Bangalore": "Bengaluru"})

In [ ]:
# Filter invalid records (e.g., created_at after slot_at)
df_all_legacy["created_at"] = pd.to_datetime(df_all_legacy["created_at"], errors="coerce")
df_all_legacy["slot_at"] = pd.to_datetime(df_all_legacy["slot_at"], errors="coerce")

valid_dates_mask = df_all_legacy["created_at"] <= df_all_legacy["slot_at"]
df_legacy_clean = df_all_legacy[valid_dates_mask].copy()

# Save merged dataset
df_legacy_clean.to_csv(CLEAN_DIR / "bookings_merged.csv.gz", index=False, compression="gzip")
print("Saved merged bookings to data/clean/bookings_merged.csv.gz")